In [0]:
#pip install faker

In [0]:
#%restart_python

In [0]:

"""
Funções para geração de dados sintéticos de uma transportadora logística:
armazéns, clientes, produtos, funcionários e vendas — além do cálculo de
distâncias reais (via OpenRouteService) e roteirização simples por
vizinho mais próximo.

Observação: o projeto é uma homenagem à sitcom "The Office" (versão
americana) — os funcionários em monta_funcionarios() são personagens da
Dunder Mifflin (Jim Halpert, Dwight Schrute, Michael Scott, etc.), com
municípios brasileiros usados apenas para dar realismo geográfico aos
dados sintéticos (endereços, distâncias, rotas).
"""

import io
import random

import pandas as pd
import requests
from faker import Faker


# ---------------------------------------------------------------------
# Configuração inicial
# ---------------------------------------------------------------------

fake = Faker('pt_BR')
Faker.seed(42)
random.seed(42)  # garante reprodutibilidade também para random.choice/sample,
                  # não só para o Faker

URL_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
URL_LAT_LONG = "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv"


# ---------------------------------------------------------------------
# Municípios e geolocalização
# ---------------------------------------------------------------------

def carrega_municipios():
    """
    Busca todos os municípios do Brasil na API do IBGE e retorna um
    DataFrame já achatado (sem colunas aninhadas), com o nome, UF,
    código da UF e região de cada um.
    """
    response = requests.get(URL_IBGE)
    response.raise_for_status()
    dados_json = response.json()

    df = pd.json_normalize(dados_json)
    df = df[[
        "id",
        "nome",
        "microrregiao.mesorregiao.UF.sigla",
        "microrregiao.mesorregiao.UF.id",
        "microrregiao.mesorregiao.UF.regiao.nome",
    ]]

    df.columns = ["codigo_ibge", "nome_cidade", "uf", "sigla_id", "regiao"]
    return df


def carrega_lat_long():
    """
    Baixa o CSV com latitude/longitude de todos os municípios brasileiros
    (fonte: kelvins/municipios-brasileiros) e retorna só as colunas
    necessárias para geolocalização.

    Usa um User-Agent customizado porque o GitHub às vezes bloqueia (403)
    requisições sem esse cabeçalho.
    """
    resp = requests.get(URL_LAT_LONG, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()

    df = pd.read_csv(io.StringIO(resp.text))

    # inclui codigo_ibge — usado para casar com df_municipios sem
    # depender de nome/UF (evita quebra por divergência de acentuação)
    return df[['codigo_ibge', 'nome', 'codigo_uf', 'latitude', 'longitude']]


def buscar_lat_long(df_lat_long, codigo_ibge):
    """
    Retorna (latitude, longitude) de uma cidade específica, buscando
    pelo código IBGE — mais robusto que buscar por nome, já que evita
    divergências de acentuação/grafia entre fontes de dados diferentes
    (ex: API do IBGE vs CSV do kelvins/municipios-brasileiros).
    """
    linha = df_lat_long[df_lat_long['codigo_ibge'] == codigo_ibge].head(1)

    if linha.empty:
        raise ValueError(f"Código IBGE não encontrado em df_lat_long: {codigo_ibge}")

    return linha['latitude'].values[0], linha['longitude'].values[0]


# ---------------------------------------------------------------------
# Entidades de negócio
# ---------------------------------------------------------------------

def montar_armazens(df_municipios, df_lat_long):
    """
    Monta os armazéns fixos da transportadora, um em cada cidade da lista
    abaixo, com código, localização e coordenadas.

    Recebe df_municipios e df_lat_long como parâmetros (em vez de buscar
    internamente) para evitar repetir chamadas de API desnecessárias
    quando montar_clientes() também precisar dos mesmos dados.
    """
    #Adição do armazém de Goiania - 03/08/26
    armazem_cidades = ['São Paulo', 'Curitiba', 'Recife', 'Belém','Goiânia']
    armazem_codigo = ['A1', 'A2', 'A3', 'A4','A5']

    armazens = []

    for i, cidade in enumerate(armazem_cidades):
        linha = df_municipios[df_municipios['nome_cidade'] == cidade].head(1)

        if linha.empty:
            raise ValueError(f"Cidade não encontrada em df_municipios: {cidade}")

        codigo_ibge = linha['codigo_ibge'].values[0]
        uf = linha['uf'].values[0]
        regiao = linha['regiao'].values[0]

        latitude, longitude = buscar_lat_long(df_lat_long, codigo_ibge)

        armazens.append({
            'id': i + 1,
            'codigo': armazem_codigo[i],
            'cidade': cidade,
            'uf': uf,
            'regiao': regiao,
            'latitude': latitude,
            'longitude': longitude,
        })

    return pd.DataFrame(armazens)


def montar_produtos():
    """Catálogo fixo de produtos (papelaria) com peso e valor unitário."""

    produtos = [
        [1, 'Papel Kraft', 3, 80],
        [2, 'Papel-Cartão', 20, 250],
        [3, 'Papel Supremo', 20, 220],
        [4, 'Papel Sulfite', 23, 300],
        [5, 'Papel Couché', 6, 200],
        [6, 'Papel Pólen', 13, 220],
        [7, 'Papel Color Plus', 2.8, 100],
        [8, 'Papel Vegetal', 2, 80],
        [9, 'Papel Fotográfico', 1.2, 40],
    ]

    return pd.DataFrame(
        produtos,
        columns=['id', 'produto', 'peso', 'valor_unitario']
    )


def montar_clientes(df_municipios, df_lat_long, quantidade=50):
    """
    Sorteia `quantidade` municípios aleatórios e gera um cliente fictício
    (empresa) para cada um, com dados de contato via Faker.
    """
    clientes = []

    for x in range(quantidade):
        linha_aleatoria = df_municipios.sample(n=1)

        nome_cidade = linha_aleatoria['nome_cidade'].values[0]
        codigo_ibge = linha_aleatoria['codigo_ibge'].values[0]
        regiao = linha_aleatoria['regiao'].values[0]
        uf = linha_aleatoria['uf'].values[0]

        latitude, longitude = buscar_lat_long(df_lat_long, codigo_ibge)

        clientes.append({
            'id': x + 1,
            'name': fake.company(),
            'endereco_entrega': fake.street_address(),
            'cidade': nome_cidade,
            'uf': uf,
            'regiao': regiao,
            'latitude': latitude,
            'longitude': longitude,
            'email': fake.ascii_company_email(),
            'telefone': fake.phone_number(),
        })

    return pd.DataFrame(clientes)


def monta_funcionarios():
    """Quadro fixo de funcionários, cada um com uma área e data de contratação aleatória."""

    funcionarios = [
        [2, 'Jim Halpert', 'Vendas'],
        [1, 'Dwight Schrute', 'Vendas'],
        [3, 'Stanley Hudson', 'Vendas'],
        [4, 'Phyllis Vance', 'Vendas'],
        [5, 'Andy Bernard', 'Vendas'],
        [6, 'Ryan Howard', 'Vendas'],
        [7, 'Michael Scott', 'Gerencia'],
        [8, 'Pamela Halpert', 'Administrativo'],
        [9, 'Angela Martin', 'Contabilidade'],
        [10, 'Oscar Martinez', 'Contabilidade'],
        [11, 'Kevin Malone', 'Contabilidade'],
        [12, 'Kelly Kapoor', 'Administrativo'],
        [13, 'Creed Bratton', 'Administrativo'],
        [14, 'Meredith Palmer', 'Administrativo'],
        [15, 'Darryl Philbin', 'Armazém'],
        [16, 'Lonny Smith', 'Armazém'],
        [17, 'Madge Parker', 'Armazém'],
        [18, 'Glenn Godwin', 'Armazém'],
        [19, 'Hide Lee', 'Armazém'],
        [20, 'Toby Flenderson', 'Recursos Humanos'],
    ]

    datas_possiveis = pd.date_range(
        start="2020-01-01",
        end="2022-05-10"
    )

    registros = []

    for id_func, nome, area in funcionarios:
        registros.append({
            'id': id_func,
            'Nome': nome,
            'Area': area,
            'Data_Contratacao': random.choice(datas_possiveis),
        })

    return pd.DataFrame(registros)


def montar_vendas(df_produtos, df_clientes, df_funcionarios, datas, n):
    """
    Gera `n` vendas sintéticas, sorteando produto, cliente, vendedor
    (apenas funcionários da área "Vendas") e data.
    """

    # calculado uma única vez fora do loop, já que não muda a cada venda
    vendedores = df_funcionarios[
        df_funcionarios['Area'] == 'Vendas'
    ]['id'].tolist()

    vendas = []

    for x in range(n):
        produto = df_produtos.sample(n=1).iloc[0]
        qtd = random.randint(1, 30)

        vendas.append({
            'id_venda': x + 1,
            'data_venda': random.choice(datas),
            'id_cliente': random.choice(df_clientes['id']),
            'id_vendedor': random.choice(vendedores),
            'produto_id': produto['id'],
            'quantidade': qtd,
            'valor_total': qtd * produto['valor_unitario'],
        })

    return pd.DataFrame(vendas)


# ---------------------------------------------------------------------
# Distâncias (OpenRouteService) e atribuição de armazém
# ---------------------------------------------------------------------

def calcula_distancias(df_armazem, df_clientes, api_key):
    """
    Calcula a distância rodoviária entre cada armazém e cada cliente
    via OpenRouteService Matrix API, em uma única chamada.

    Usa "sources"/"destinations" para pedir apenas a matriz armazém→cliente,
    evitando gastar cota da API com pares armazém↔armazém ou cliente↔cliente.
    """

    locations_armazem = df_armazem[['longitude', 'latitude']].values.tolist()
    locations_cliente = df_clientes[['longitude', 'latitude']].values.tolist()
    locations = locations_armazem + locations_cliente

    n_armazens = len(locations_armazem)
    n_clientes = len(locations_cliente)

    url_matrix = "https://api.openrouteservice.org/v2/matrix/driving-car"

    headers = {
        'Authorization': api_key,
        'Content-Type': 'application/json',
    }

    body = {
        "locations": locations,
        "sources": list(range(n_armazens)),
        "destinations": list(range(n_armazens, n_armazens + n_clientes)),
        "metrics": ["distance", "duration"],
    }

    response = requests.post(url_matrix, json=body, headers=headers)
    response.raise_for_status()

    dados = response.json()

    if 'distances' not in dados:
        raise RuntimeError(f"Erro na API do OpenRouteService: {dados}")

    registros_distancia = []

    for i in range(n_armazens):
        for j in range(n_clientes):
            duracao_s = dados['durations'][i][j] if 'durations' in dados else None

            registros_distancia.append({
                'id_armazem': df_armazem.iloc[i]['id'],
                'id_cliente': df_clientes.iloc[j]['id'],
                'distancia_metros': dados['distances'][i][j],
                'duracao_min': round(duracao_s / 60) if duracao_s is not None else None,
            })

    return pd.DataFrame(registros_distancia)

def atribui_armazem(df_armazem, df_clientes, api_key):
    """
    Para cada cliente, encontra o armazém mais próximo.

    Faz o cross join armazém × cliente, junta com as distâncias calculadas
    e mantém, por cliente, apenas a linha com a menor distância.
    """
    perto = 500
    medio = 800

    REGIOES_PERMITIDAS = {
        'Norte': ['Norte', 'Nordeste'],
        'Nordeste': ['Nordeste', 'Norte'],
        'Centro-Oeste': ['Centro-Oeste', 'Sudeste'],
        'Sudeste': ['Sudeste', 'Sul'],
        'Sul': ['Sul', 'Sudeste'],
    }

    df_distancias = calcula_distancias(df_armazem, df_clientes, api_key)

    df = df_armazem.merge(df_clientes, how='cross', suffixes=('_armazem', '_cliente'))

    df = df[df.apply(
        lambda linha: linha['regiao_armazem'] in REGIOES_PERMITIDAS[linha['regiao_cliente']],
        axis=1
    )]

    df = df.merge(
        df_distancias,
        on=['id_armazem', 'id_cliente'],
    ).drop(
        columns=[
            'codigo', 'cidade_armazem', 'latitude_armazem', 'longitude_armazem',
            'name', 'endereco_entrega', 'cidade_cliente', 'uf_cliente',
            'latitude_cliente', 'longitude_cliente', 'email', 'telefone',
            'regiao_armazem', 'regiao_cliente',
        ]
    ).reset_index(drop=True)

    df['distancia_km'] = (df['distancia_metros'] / 1000).round(2)
    df = df.drop(columns=['distancia_metros'])

    # remove linhas sem distância válida antes de calcular o mínimo
    df_validas = df.dropna(subset=['distancia_km'])

    # define o prazo de entrega com base na distância
    # até 500 km → 1 dia
    # de 500 a 800 km → 2 dias
    # acima de 800 km → 3 dias
    df['prazo'] = 3

    df.loc[df['distancia_km'] <= perto, 'prazo'] = 1

    df.loc[
        (df['distancia_km'] > perto) &
        (df['distancia_km'] <= medio),
        'prazo'
    ] = 2

    indice_menor_distancia = df_validas.groupby('id_cliente')['distancia_km'].idxmin()
    return df_validas.loc[indice_menor_distancia].reset_index(drop=True)


def calcula_distancias_clientes(df_clientes, df_entrega, api_key):
    """
    Calcula a distância e duração rodoviária entre cada par de clientes,
    separadamente para cada armazém — usando só os clientes atribuídos
    a ele (matriz cliente × cliente por grupo, excluindo a diagonal
    onde origem == destino).

    Evita calcular distância entre clientes que nunca estarão na mesma
    rota (ex: cliente do armazém de Belém x cliente do armazém de
    Curitiba), reduzindo bastante o número de pares em relação a uma
    matriz global de todos contra todos.

    Necessário para a roteirização: depois da primeira entrega, o
    caminho segue de cliente em cliente, não volta ao armazém.
    """
    url_matrix = "https://api.openrouteservice.org/v2/matrix/driving-car"

    headers = {
        'Authorization': api_key,
        'Content-Type': 'application/json',
    }

    registros = []

    for id_armazem, grupo in df_entrega.groupby('id_armazem'):
        ids_clientes = grupo['id_cliente'].unique()

        # filtra só os clientes desse armazém, e remove quem não tem
        # coordenadas válidas
        df = df_clientes[df_clientes['id'].isin(ids_clientes)]
        df = df.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

        # se sobrar 1 ou 0 clientes válidos nesse armazém, não há par
        # possível para calcular — pula para o próximo grupo
        if len(df) < 2:
            continue

        locations = df[['longitude', 'latitude']].values.tolist()

        body = {
            "locations": locations,
            "metrics": ["distance", "duration"],
        }

        response = requests.post(url_matrix, json=body, headers=headers)
        response.raise_for_status()

        dados = response.json()

        if 'distances' not in dados or 'durations' not in dados:
            raise RuntimeError(f"Erro na API: {dados}")

        for i, origem in df.iterrows():
            for j, destino in df.iterrows():
                if i == j:
                    continue

                distancia = dados['distances'][i][j]
                duracao = dados['durations'][i][j]

                registros.append({
                    'id_cliente_origem': origem['id'],
                    'id_cliente_destino': destino['id'],
                    'distancia_km': round(distancia / 1000, 2) if distancia is not None else None,
                    'duracao_min': round(duracao / 60) if duracao is not None else None,
                })

    return pd.DataFrame(registros)


def calcular_melhor_rota(df_entregas_grupo, df_dist_clientes):
    """
    Ordena as entregas de um grupo (um armazém + uma data de saída) usando
    a heurística do vizinho mais próximo: parte do cliente mais perto do
    armazém e, a cada passo, segue para o cliente não visitado mais
    próximo do ponto atual.

    Não é a rota matematicamente ótima (isso seria um problema de TSP),
    mas é uma aproximação simples e rápida de calcular.

    Parâmetros
    ----------
    df_entregas_grupo : DataFrame
        Linhas de UM armazém + UMA data, com colunas id_cliente e
        distancia_km (armazém -> cliente).

    df_dist_clientes : DataFrame
        Resultado de calcula_distancias_clientes(), com a distância
        entre cada par de clientes.

    Retorno
    -------
    DataFrame com id_cliente, ordem_entrega (1, 2, 3...) e
    distancia_percorrida (distância do ponto anterior até aquela parada
    — não é acumulada).
    """

    # um cliente pode ter mais de uma venda no mesmo dia, mas só é
    # visitado uma vez fisicamente
    df_entregas_grupo = (
        df_entregas_grupo
        .dropna(subset=['id_cliente'])
        .drop_duplicates(subset=['id_cliente'])
    )

    clientes_restantes = df_entregas_grupo['id_cliente'].tolist()

    if not clientes_restantes:
        return pd.DataFrame()

    # ponto de partida: cliente mais próximo do armazém
    primeira_linha = df_entregas_grupo.sort_values('distancia_km').iloc[0]
    cliente_atual = primeira_linha['id_cliente']

    ordem = [cliente_atual]
    distancias = [primeira_linha['distancia_km']]
    duracoes = [primeira_linha['duracao_min']]

    clientes_restantes.remove(cliente_atual)

    # a cada passo, vai para o cliente restante mais próximo do atual
    while clientes_restantes:
        candidatos = df_dist_clientes[
            (df_dist_clientes['id_cliente_origem'] == cliente_atual) &
            (df_dist_clientes['id_cliente_destino'].isin(clientes_restantes))
        ].dropna(subset=['distancia_km'])

        # se não houver conexão válida, interrompe a rota
        if candidatos.empty:
            print(f"Sem rota encontrada para cliente {cliente_atual}")
            break

        proxima_linha = candidatos.sort_values('distancia_km').iloc[0]

        proximo_cliente = proxima_linha['id_cliente_destino']

        # segurança contra NaN
        if pd.isna(proximo_cliente):
            break

        ordem.append(proximo_cliente)
        distancias.append(proxima_linha['distancia_km'])
        duracoes.append(proxima_linha['duracao_min'])

        clientes_restantes.remove(proximo_cliente)
        cliente_atual = proximo_cliente

    return pd.DataFrame({
        'id_cliente': ordem,
        'ordem_entrega': range(1, len(ordem) + 1),
        'distancia_percorrida_km': distancias,
        'duracao_percorrida_min': duracoes,
    })


In [0]:
import os
import pandas as pd


def salvar_parquet(df, caminho):
    """Salva um DataFrame em Parquet no caminho informado, com timestamps
    em microssegundos (Spark não lê timestamps em nanossegundos)."""
    df.to_parquet(caminho, index=False, coerce_timestamps='us', allow_truncated_timestamps=True)


def carregar_parquet(caminho):
    """Carrega um DataFrame salvo em Parquet, ou None se o arquivo não existir."""
    if not os.path.exists(caminho):
        return None
    return pd.read_parquet(caminho)


def carregar_ou_gerar(caminho, funcao_geradora, forcar_atualizacao=False):
    """
    Carrega um DataFrame do cache (Parquet), se existir; caso contrário
    (ou se forcar_atualizacao=True), executa `funcao_geradora`, salva o
    resultado em disco e retorna.
    """
    if not forcar_atualizacao:
        df_em_cache = carregar_parquet(caminho)
        if df_em_cache is not None:
            return df_em_cache

    df = funcao_geradora()
    salvar_parquet(df, caminho)
    return df

In [0]:
"""
Script principal: gera a base sintética completa da transportadora
(homenagem a The Office) e calcula a rota otimizada de entrega para
cada armazém, em cada data de saída.

Os resultados intermediários e finais são cacheados em Parquet, nos
Volumes do Unity Catalog (/Volumes/workspace/dundermiffin/raw e
/Volumes/workspace/dundermiffin/logistica), para não depender de
reprocessar tudo (e regastar cota de API) a cada execução.
Use FORCAR_ATUALIZACAO = True para ignorar o cache e regenerar tudo do zero.
"""

DIR_RAW = "/Volumes/workspace/dundermiffin/raw" #dado externo, cru, estável (não muda de execução pra execução)
DIR_CACHE = "/Volumes/workspace/dundermiffin/cache" #resultado caro de recalcular (API com rate limit), mas sem valor analítico isolado — puramente otimização técnica
DIR_SOURCE = "/Volumes/workspace/dundermiffin/source" #entidades de negócio geradas, reutilizáveis por qualquer área futura (dim)
DIR_LOGISTICA = "/Volumes/workspace/dundermiffin/logistica" #fatos reais e exclusivos dessa etapa de negócio


# se True, ignora qualquer cache existente nos Volumes e regenera tudo do zero
FORCAR_ATUALIZACAO = True

# ---------------------------------------------------------------------
# Configuração
# ---------------------------------------------------------------------

# Usa a chave API do query parameter configurado no notebook
api_key = dbutils.widgets.get('api_key')

# período em que as vendas serão sorteadas, e quantidade de vendas a gerar
datas = pd.date_range(start="2026-01-01", end="2026-07-30")
n = 2000

# mostra todas as linhas ao exibir DataFrames grandes (sem truncar com "...")
pd.set_option('display.max_rows', None)


# ---------------------------------------------------------------------
# 1. Geração das entidades base (com cache em Parquet nos Volumes)
# ---------------------------------------------------------------------

# dados "de base" (mudam raramente) ficam em /Volumes/.../raw
df_municipios = carregar_ou_gerar(
    f"{DIR_RAW}/municipios.parquet", carrega_municipios, FORCAR_ATUALIZACAO
)
df_lat_long = carregar_ou_gerar(
    f"{DIR_RAW}/lat_long.parquet", carrega_lat_long, FORCAR_ATUALIZACAO
)

# entidades sintéticas da etapa de logística ficam em /Volumes/.../logistica
df_armazem = carregar_ou_gerar(
    f"{DIR_SOURCE}/armazens.parquet",
    lambda: montar_armazens(df_municipios, df_lat_long),
    FORCAR_ATUALIZACAO,
)
df_produtos = carregar_ou_gerar(
    f"{DIR_SOURCE}/produtos.parquet", montar_produtos, FORCAR_ATUALIZACAO
)
df_clientes = carregar_ou_gerar(
    f"{DIR_SOURCE}/clientes.parquet",
    lambda: montar_clientes(df_municipios, df_lat_long),
    FORCAR_ATUALIZACAO,
)
df_funcionarios = carregar_ou_gerar(
    f"{DIR_SOURCE}/funcionarios.parquet", monta_funcionarios, FORCAR_ATUALIZACAO
)
df_vendas = carregar_ou_gerar(
    f"{DIR_SOURCE}/vendas.parquet",
    lambda: montar_vendas(df_produtos, df_clientes, df_funcionarios, datas, n),
    FORCAR_ATUALIZACAO,
)

# atribui a cada cliente o armazém mais próximo (distância real via OpenRouteService)
# — envolve chamada de API, por isso vale sempre cachear
df_entrega = carregar_ou_gerar(
    f"{DIR_LOGISTICA}/entrega.parquet",
    lambda: atribui_armazem(df_armazem, df_clientes, api_key),
    FORCAR_ATUALIZACAO,
)


# ---------------------------------------------------------------------
# 2. Junta entregas com vendas para descobrir a data de saída de cada rota
# ---------------------------------------------------------------------

# cada venda gera uma entrega no dia seguinte à compra
df_entrega_completo = df_entrega.merge(
    df_vendas,
    how='inner',
    on=['id_cliente'],
)
df_entrega_completo['data_saida'] = df_entrega_completo['data_venda'] + pd.Timedelta(days=1)


# ---------------------------------------------------------------------
# 3. Roteirização: ordem de visita dos clientes por armazém + data
# ---------------------------------------------------------------------
"""
 Distância entre cada par de clientes, usada para montar a rota depois
 da primeira entrega (o caminho segue de cliente em cliente, não volta
 ao armazém a cada parada). É a chamada mais cara de refazer, por isso
 é sempre cacheada.
"""
df_dist_clientes = carregar_ou_gerar(
    f"{DIR_CACHE}/distancias_cliente_cliente.parquet",
    lambda: calcula_distancias_clientes(df_clientes, df_entrega, api_key),
    FORCAR_ATUALIZACAO,
)

"""
 Calcula a melhor rota (vizinho mais próximo) separadamente para cada
 combinação de armazém + data de saída
"""
rotas_por_grupo = []
for (id_armazem, data_saida), grupo in df_entrega_completo.groupby(['id_armazem', 'data_saida']):
    rota = calcular_melhor_rota(grupo, df_dist_clientes)
    rota['id_armazem'] = id_armazem
    rota['data_saida'] = data_saida
    rotas_por_grupo.append(rota)

df_rotas_otimizadas = pd.concat(rotas_por_grupo, ignore_index=True)
"""
 Ordena o resultado final: por data, depois por armazém, depois pela
 sequência de entrega dentro de cada rota
"""
df_rotas = df_rotas_otimizadas.sort_values(['data_saida', 'id_armazem', 'ordem_entrega'])

# persiste o resultado final da rota — é o output principal dessa análise
salvar_parquet(df_rotas, f"{DIR_LOGISTICA}/rotas_otimizadas.parquet")
